In [2]:
# ============================================================
# RAG WITH GPT4ALL DESKTOP
# ============================================================
# This notebook demonstrates a complete RAG application using:
#
# 1. PDF documents
# 2. Gemini Embeddings
# 3. ChromaDB Vector Database
# 4. GPT4All Desktop Local API
# 5. Phi-3 Mini Instruct
#
# Architecture:
#
# PDF Documents
#      ↓
# Text Extraction
#      ↓
# Text Chunking
#      ↓
# Gemini Embeddings
#      ↓
# ChromaDB
#      ↓
# User Question
#      ↓
# Gemini Query Embedding
#      ↓
# Similarity Search
#      ↓
# Retrieved Context
#      ↓
# GPT4All Desktop API
#      ↓
# Final Answer
# ============================================================

print("RAG WITH GPT4All Desktop")
print("Environment setup started...")

RAG WITH GPT4All Desktop
Environment setup started...


In [3]:
# ============================================================
# CELL 2 - IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import json
import requests
from pathlib import Path

from pypdf import PdfReader
from reportlab.pdfgen import canvas

import chromadb

from google import genai

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
# ============================================================
# CELL 3 - PROJECT PATHS AND CONFIGURATION
# ============================================================

# Get the current project directory.
# Because Jupyter was started from:
#
# C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop
#
# Path.cwd() should point to this folder.
PROJECT_DIR = Path.cwd()

# Folder containing our PDF documents
DOCUMENTS_DIR = PROJECT_DIR / "documents"

# Folder where ChromaDB will persist the vector database
CHROMA_DIR = PROJECT_DIR / "chroma_db"

# GPT4All Desktop local API configuration
GPT4ALL_HOST = "127.0.0.1"
GPT4ALL_PORT = 4891

# Complete API endpoint used for chat completions
GPT4ALL_API_URL = (
    f"http://{GPT4ALL_HOST}:{GPT4ALL_PORT}/v1/chat/completions"
)

# Model name returned by our GPT4All Desktop server
GPT4ALL_MODEL_NAME = "Phi-3 Mini Instruct"


# ------------------------------------------------------------
# Create folders if they don't already exist
# ------------------------------------------------------------

DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("Project Directory :", PROJECT_DIR)
print("Documents Folder  :", DOCUMENTS_DIR)
print("ChromaDB Folder   :", CHROMA_DIR)
print("GPT4All API       :", GPT4ALL_API_URL)
print("GPT4All Model     :", GPT4ALL_MODEL_NAME)

Project Directory : C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop
Documents Folder  : C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop\documents
ChromaDB Folder   : C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop\chroma_db
GPT4All API       : http://127.0.0.1:4891/v1/chat/completions
GPT4All Model     : Phi-3 Mini Instruct


In [5]:
# ============================================================
# CELL 4 - TEST GPT4ALL DESKTOP LOCAL API
# ============================================================
# This cell verifies that our Python notebook can communicate
# with the GPT4All Desktop application running on Windows.
#
# GPT4All Desktop is expected to be running with:
#
#     Model : Phi-3 Mini Instruct
#     Host  : 127.0.0.1
#     Port  : 4891
#
# The API endpoint is:
#
#     http://127.0.0.1:4891/v1/chat/completions
# ============================================================


# ------------------------------------------------------------
# First, check whether the GPT4All Desktop server is reachable
# ------------------------------------------------------------

MODELS_API_URL = f"http://{GPT4ALL_HOST}:{GPT4ALL_PORT}/v1/models"

try:
    response = requests.get(
        MODELS_API_URL,
        timeout=10
    )

    print("HTTP Status Code:", response.status_code)

    if response.status_code == 200:
        print("GPT4All Desktop server is reachable!")
    else:
        print("GPT4All Desktop returned an unexpected status.")

except requests.exceptions.RequestException as e:
    print("Could not connect to GPT4All Desktop.")
    print("Error:", e)


# ------------------------------------------------------------
# Display the models available through GPT4All Desktop
# ------------------------------------------------------------

if response.status_code == 200:

    models_data = response.json()

    print("\nAvailable model(s):")

    for model in models_data.get("data", []):
        print("-", model.get("id"))

HTTP Status Code: 200
GPT4All Desktop server is reachable!

Available model(s):
- Phi-3 Mini Instruct


In [6]:
# ============================================================
# CELL 5 - TEST GPT4ALL DESKTOP CHAT COMPLETION API
# ============================================================
# This cell sends a real question to the Phi-3 Mini Instruct
# model running inside GPT4All Desktop.
#
# This confirms that:
#
# Python
#    ↓
# GPT4All Desktop API
#    ↓
# Phi-3 Mini Instruct
#    ↓
# Response
#
# is working correctly.
# ============================================================


# ------------------------------------------------------------
# Test question
# ------------------------------------------------------------

test_question = "What is Azure Functions?"


# ------------------------------------------------------------
# Create the OpenAI-compatible API request
# ------------------------------------------------------------

payload = {
    "model": GPT4ALL_MODEL_NAME,

    "messages": [
        {
            "role": "user",
            "content": test_question
        }
    ],

    # Maximum number of tokens in the generated answer
    "max_tokens": 200,

    # Lower temperature gives more consistent answers
    "temperature": 0.2
}


# ------------------------------------------------------------
# Send request to GPT4All Desktop
# ------------------------------------------------------------

try:

    response = requests.post(
        GPT4ALL_API_URL,
        json=payload,
        timeout=120
    )

    print("HTTP Status Code:", response.status_code)

    response.raise_for_status()

    result = response.json()


    # --------------------------------------------------------
    # Extract the generated answer
    # --------------------------------------------------------

    answer = result["choices"][0]["message"]["content"]

    print("\nQuestion:")
    print(test_question)

    print("\nGPT4All Desktop Answer:")
    print(answer)


except requests.exceptions.RequestException as e:

    print("GPT4All Desktop API request failed.")
    print("Error:", e)

except (KeyError, IndexError) as e:

    print("Unexpected response format from GPT4All Desktop.")
    print("Error:", e)

HTTP Status Code: 200

Question:
What is Azure Functions?

GPT4All Desktop Answer:
 Azure Functions is a serverless compute service provided by Microsoft as part of its cloud computing platform, Azure. It enables developers to build and deploy scalable applications that respond in real-time without managing infrastructure. Here are some key aspects of Azure Functions:

1. Serverless Computing Model: Unlike traditional models where you have to provision servers or containers for your application's runtime environment, Azure Functions abstract away the underlying infrastructure and automatically scales up/down based on demand. This means developers can focus solely on writing code without worrying about managing server resources.

2. Multiple Programming Languages: Azure Functions supports several programming languages such as C#, F# (using .NET), JavaScript (.NET, Node.js or Python runtime), Java and PowerShell for building functions that respond to various events like HTTP requests, ti

In [7]:
# ============================================================
# CELL 6 - CONFIGURE GEMINI API
# ============================================================
# Gemini is used in this project for:
#
#     Text → Embedding Vector
#
# GPT4All Desktop is used separately for:
#
#     Context + Question → Final Answer
#
# Therefore:
#
# Gemini                → Embeddings
# GPT4All Desktop       → Answer Generation
# ============================================================


# ------------------------------------------------------------
# Read the API key from the Windows environment variable
# ------------------------------------------------------------

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


# ------------------------------------------------------------
# Validate the API key
# ------------------------------------------------------------

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found.\n"
        "Please set it as a Windows environment variable "
        "and restart Jupyter."
    )


# ------------------------------------------------------------
# Create the Gemini client
# ------------------------------------------------------------

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


print("Gemini API key loaded successfully!")
print("Gemini client created successfully!")

Gemini API key loaded successfully!
Gemini client created successfully!


In [8]:
# ============================================================
# CELL 7 - TEST GEMINI EMBEDDING MODEL
# ============================================================
# Gemini will be used to convert text into numerical vectors.
#
# These vectors are the foundation of our semantic search.
# Later, the vectors will be stored in ChromaDB.
# ============================================================


# ------------------------------------------------------------
# Test text
# ------------------------------------------------------------

test_text = (
    "Azure Functions is a serverless compute service "
    "that runs event-driven code."
)


# ------------------------------------------------------------
# Generate the embedding
# ------------------------------------------------------------

embedding_response = gemini_client.models.embed_content(
    model="gemini-embedding-001",
    contents=test_text
)


# ------------------------------------------------------------
# Extract the embedding vector
# ------------------------------------------------------------

test_embedding = embedding_response.embeddings[0].values


# ------------------------------------------------------------
# Display basic information
# ------------------------------------------------------------

print("Gemini embedding generated successfully!")
print("Embedding dimensions:", len(test_embedding))
print("First 5 values:", test_embedding[:5])

Gemini embedding generated successfully!
Embedding dimensions: 3072
First 5 values: [0.011270939, 0.0058875238, 0.019500334, -0.074818365, -0.021805186]


In [9]:
# ============================================================
# CELL 8 - CREATE AZURE FUNCTIONS KNOWLEDGE PDF
# ============================================================
# This PDF will be our first knowledge-base document.
#
# It contains information about Azure Functions that our RAG
# application can retrieve when answering user questions.
# ============================================================


# ------------------------------------------------------------
# PDF file path
# ------------------------------------------------------------

azure_functions_pdf = DOCUMENTS_DIR / "azure_functions.pdf"


# ------------------------------------------------------------
# Create the PDF
# ------------------------------------------------------------

pdf = canvas.Canvas(str(azure_functions_pdf))

# Page size: approximately 8.5 x 11 inches
page_width = 612
page_height = 792

# Starting position
x = 50
y = 750

# Title
pdf.setFont("Helvetica-Bold", 20)
pdf.drawString(x, y, "Azure Functions")

y -= 35

# Subtitle
pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "Serverless Compute on Microsoft Azure")

y -= 30


# ------------------------------------------------------------
# Helper function for writing paragraphs
# ------------------------------------------------------------

def write_paragraph(pdf, text, x, y, max_width=510, line_height=16):
    """
    Write a paragraph to the PDF and automatically wrap
    long lines.
    """

    pdf.setFont("Helvetica", 10)

    words = text.split()
    line = ""

    for word in words:

        test_line = line + " " + word if line else word

        if pdf.stringWidth(test_line, "Helvetica", 10) <= max_width:
            line = test_line
        else:
            pdf.drawString(x, y, line)
            y -= line_height
            line = word

    if line:
        pdf.drawString(x, y, line)
        y -= line_height

    return y


# ------------------------------------------------------------
# Section 1
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "1. What is Azure Functions?")

y -= 22

text = (
    "Azure Functions is a serverless compute service in Microsoft Azure "
    "that allows developers to run small pieces of code without managing "
    "the underlying server infrastructure. Functions are commonly used "
    "for event-driven applications where code executes in response to "
    "events such as HTTP requests, timers, messages, and changes in "
    "Azure Storage."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 2
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "2. Triggers")

y -= 22

text = (
    "A trigger defines the event that causes an Azure Function to execute. "
    "Common triggers include HTTP triggers, Timer triggers, Azure Blob "
    "Storage triggers, Azure Queue Storage triggers, and Azure Service "
    "Bus triggers. An HTTP trigger can expose a function through a REST "
    "endpoint."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 3
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "3. Bindings")

y -= 22

text = (
    "Bindings provide a way for an Azure Function to connect to other "
    "Azure services without writing extensive connection code. Input "
    "bindings provide data to a function, while output bindings send "
    "data to another service. Examples include bindings for Blob Storage, "
    "Queue Storage, Cosmos DB, and Service Bus."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 4
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "4. Hosting Plans")

y -= 22

text = (
    "Azure Functions can run using different hosting options. Consumption "
    "plans automatically scale based on demand and charge according to "
    "execution and resource usage. Premium hosting provides additional "
    "features such as pre-warmed instances and more predictable performance. "
    "Dedicated hosting runs functions on dedicated App Service infrastructure."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 5
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "5. Durable Functions")

y -= 22

text = (
    "Durable Functions is an extension of Azure Functions that helps "
    "developers implement stateful workflows in a serverless environment. "
    "It supports orchestrator functions, activity functions, and durable "
    "entities. Durable Functions can be used for long-running processes, "
    "function chaining, fan-out and fan-in patterns, and human interaction "
    "workflows."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 6
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "6. Common Use Cases")

y -= 22

text = (
    "Common Azure Functions scenarios include REST APIs, scheduled jobs, "
    "file processing, message processing, event-driven automation, "
    "background tasks, data transformation, and integration with Azure "
    "services such as Event Grid, Service Bus, Blob Storage, and Cosmos DB."
)

y = write_paragraph(pdf, text, x, y)


# ------------------------------------------------------------
# Save PDF
# ------------------------------------------------------------

pdf.save()


print("Azure Functions PDF created successfully!")
print("File:", azure_functions_pdf)
print("Exists:", azure_functions_pdf.exists())

Azure Functions PDF created successfully!
File: C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop\documents\azure_functions.pdf
Exists: True


In [10]:
# ============================================================
# CELL 9 - CREATE AZURE LOGIC APPS KNOWLEDGE PDF
# ============================================================
# This is our second knowledge-base document.
#
# The RAG system will retrieve information from this PDF when
# answering questions about Azure Logic Apps.
# ============================================================


# ------------------------------------------------------------
# PDF file path
# ------------------------------------------------------------

azure_logic_apps_pdf = DOCUMENTS_DIR / "azure_logic_apps.pdf"


# ------------------------------------------------------------
# Create the PDF
# ------------------------------------------------------------

pdf = canvas.Canvas(str(azure_logic_apps_pdf))

x = 50
y = 750

# Title
pdf.setFont("Helvetica-Bold", 20)
pdf.drawString(x, y, "Azure Logic Apps")

y -= 35

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "Workflow Automation and Integration")

y -= 30


# ------------------------------------------------------------
# Section 1
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "1. What are Azure Logic Apps?")

y -= 22

text = (
    "Azure Logic Apps is a cloud service in Microsoft Azure used to "
    "create automated workflows that integrate applications, services, "
    "systems, and data. Logic Apps provide a visual workflow designer "
    "and a large collection of connectors for communicating with external "
    "and Azure services."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 2
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "2. Workflows")

y -= 22

text = (
    "A Logic App workflow consists of a trigger followed by one or more "
    "actions. The trigger starts the workflow when an event occurs. "
    "Actions perform operations such as calling an API, sending an email, "
    "reading a database record, transforming data, or sending a message "
    "to a queue."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 3
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "3. Triggers")

y -= 22

text = (
    "Logic Apps support many trigger types. A workflow can start when an "
    "HTTP request is received, when a scheduled time is reached, when a "
    "message arrives, when a file is created, or when another connected "
    "service produces an event."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 4
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "4. Connectors")

y -= 22

text = (
    "Connectors allow Logic Apps to communicate with other applications "
    "and services. Microsoft provides connectors for services such as "
    "Azure Service Bus, Blob Storage, SQL Server, Microsoft 365, SharePoint, "
    "Salesforce, and many other systems. Connectors simplify integration "
    "by providing predefined operations."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 5
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "5. Azure Logic Apps Standard and Consumption")

y -= 22

text = (
    "Azure Logic Apps provides Standard and Consumption hosting options. "
    "Consumption workflows are commonly used for individual workflows "
    "that use a multitenant model. Standard supports workflows using a "
    "single-tenant model and provides additional capabilities for hosting "
    "and managing workflows. The appropriate option depends on workload, "
    "integration, networking, and operational requirements."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 6
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "6. Common Use Cases")

y -= 22

text = (
    "Logic Apps are commonly used for enterprise integration, business "
    "process automation, file processing, API orchestration, notification "
    "workflows, data movement, message processing, and connecting cloud "
    "services with on-premises systems."
)

y = write_paragraph(pdf, text, x, y)

y -= 10


# ------------------------------------------------------------
# Section 7
# ------------------------------------------------------------

pdf.setFont("Helvetica-Bold", 13)
pdf.drawString(x, y, "7. Logic Apps and Azure Functions")

y -= 22

text = (
    "Logic Apps and Azure Functions can be used together. Logic Apps are "
    "well suited to workflow orchestration and service integration, while "
    "Azure Functions are suited to executing custom code in response to "
    "events. A Logic App can call an Azure Function when custom processing "
    "or specialized business logic is required."
)

y = write_paragraph(pdf, text, x, y)


# ------------------------------------------------------------
# Save PDF
# ------------------------------------------------------------

pdf.save()


print("Azure Logic Apps PDF created successfully!")
print("File:", azure_logic_apps_pdf)
print("Exists:", azure_logic_apps_pdf.exists())

Azure Logic Apps PDF created successfully!
File: C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop\documents\azure_logic_apps.pdf
Exists: True


In [11]:
# ============================================================
# CELL 10 - LOAD AND EXTRACT TEXT FROM PDF DOCUMENTS
# ============================================================
# This cell:
#
# 1. Finds all PDF files in the documents folder
# 2. Opens each PDF
# 3. Extracts text page by page
# 4. Stores the filename and page number
#
# Keeping page information allows our RAG system to later
# provide source citations such as:
#
# [Source: azure_functions.pdf, Page: 1]
# ============================================================


# ------------------------------------------------------------
# Function to load PDF documents
# ------------------------------------------------------------

def load_pdfs(folder):
    """
    Load all PDF documents from the specified folder.

    Returns:
        A list of dictionaries containing:
        - filename
        - page_number
        - text
    """

    pages = []

    # Find all PDF files
    pdf_files = sorted(folder.glob("*.pdf"))

    print("PDF files found:", len(pdf_files))

    for pdf_path in pdf_files:

        print(f"\nReading: {pdf_path.name}")

        # Open the PDF
        reader = PdfReader(str(pdf_path))

        print("Number of pages:", len(reader.pages))

        # Process each page
        for page_number, page in enumerate(reader.pages, start=1):

            text = page.extract_text()

            # Handle pages where no text could be extracted
            if text is None:
                text = ""

            text = text.strip()

            # Store the page information
            pages.append({
                "filename": pdf_path.name,
                "page_number": page_number,
                "text": text
            })

    return pages


# ------------------------------------------------------------
# Load the PDFs
# ------------------------------------------------------------

pages = load_pdfs(DOCUMENTS_DIR)


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PDF LOADING COMPLETE")
print("=" * 60)

print("Total pages loaded:", len(pages))

for page in pages:

    print(
        f"{page['filename']} "
        f"→ Page {page['page_number']} "
        f"→ {len(page['text'])} characters"
    )

PDF files found: 2

Reading: azure_functions.pdf
Number of pages: 1

Reading: azure_logic_apps.pdf
Number of pages: 1

PDF LOADING COMPLETE
Total pages loaded: 2
azure_functions.pdf → Page 1 → 2055 characters
azure_logic_apps.pdf → Page 1 → 2263 characters


In [12]:
# ============================================================
# CELL 11 - INSPECT EXTRACTED PDF TEXT
# ============================================================
# This cell displays the extracted text from each PDF page.
#
# This is an important validation step before chunking.
# We want to make sure the PDF text was extracted correctly.
# ============================================================


for page in pages:

    print("=" * 80)

    print(
        f"DOCUMENT : {page['filename']}"
    )

    print(
        f"PAGE     : {page['page_number']}"
    )

    print("=" * 80)

    print(page["text"])

    print("\n")

DOCUMENT : azure_functions.pdf
PAGE     : 1
Azure Functions
Serverless Compute on Microsoft Azure
1. What is Azure Functions?
Azure Functions is a serverless compute service in Microsoft Azure that allows developers to run small pieces of
code without managing the underlying server infrastructure. Functions are commonly used for event-driven
applications where code executes in response to events such as HTTP requests, timers, messages, and changes
in Azure Storage.
2. Triggers
A trigger defines the event that causes an Azure Function to execute. Common triggers include HTTP triggers,
Timer triggers, Azure Blob Storage triggers, Azure Queue Storage triggers, and Azure Service Bus triggers. An
HTTP trigger can expose a function through a REST endpoint.
3. Bindings
Bindings provide a way for an Azure Function to connect to other Azure services without writing extensive
connection code. Input bindings provide data to a function, while output bindings send data to another service.
Examples 

In [13]:
# ============================================================
# CELL 12 - CREATE TEXT CHUNKS
# ============================================================
# RAG works better when documents are divided into smaller
# pieces called "chunks".
#
# Each chunk will keep:
#
#   1. Unique chunk ID
#   2. Original PDF filename
#   3. Page number
#   4. Chunk text
#
# We use:
#
#   Chunk size = 800 characters
#   Overlap    = 150 characters
#
# The overlap helps preserve context between neighboring chunks.
# ============================================================


# ------------------------------------------------------------
# Chunking function
# ------------------------------------------------------------

def create_chunks(text, chunk_size=800, overlap=150):
    """
    Split text into overlapping character-based chunks.

    Parameters:
        text       : Source text
        chunk_size : Maximum characters per chunk
        overlap    : Number of characters shared between chunks

    Returns:
        List of text chunks
    """

    # Clean unnecessary whitespace
    text = " ".join(text.split())

    chunks = []

    start = 0

    while start < len(text):

        # Determine the end position of this chunk
        end = start + chunk_size

        # Extract the chunk
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        # Stop if we have reached the end of the text
        if end >= len(text):
            break

        # Move forward while preserving overlap
        start = end - overlap

    return chunks


# ------------------------------------------------------------
# Create chunks for all PDF pages
# ------------------------------------------------------------

all_chunks = []

chunk_counter = 0


for page in pages:

    page_chunks = create_chunks(
        page["text"],
        chunk_size=800,
        overlap=150
    )

    for chunk in page_chunks:

        all_chunks.append({
            "chunk_id": f"chunk_{chunk_counter}",
            "filename": page["filename"],
            "page_number": page["page_number"],
            "text": chunk
        })

        chunk_counter += 1


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("=" * 60)
print("TEXT CHUNKING COMPLETE")
print("=" * 60)

print("Total chunks created:", len(all_chunks))

print("\nChunks by document:")

for filename in sorted(
    set(chunk["filename"] for chunk in all_chunks)
):

    count = sum(
        1
        for chunk in all_chunks
        if chunk["filename"] == filename
    )

    print(f"- {filename}: {count} chunks")

TEXT CHUNKING COMPLETE
Total chunks created: 7

Chunks by document:
- azure_functions.pdf: 3 chunks
- azure_logic_apps.pdf: 4 chunks


In [14]:
# ============================================================
# CELL 13 - INSPECT CREATED CHUNKS
# ============================================================
# Display the chunks with their source information.
# This helps us verify that our metadata is being preserved.
# ============================================================


for chunk in all_chunks:

    print("=" * 80)

    print("Chunk ID :", chunk["chunk_id"])
    print("Source  :", chunk["filename"])
    print("Page    :", chunk["page_number"])
    print("Length  :", len(chunk["text"]))

    print("-" * 80)

    print(chunk["text"])

    print()

Chunk ID : chunk_0
Source  : azure_functions.pdf
Page    : 1
Length  : 799
--------------------------------------------------------------------------------
Azure Functions Serverless Compute on Microsoft Azure 1. What is Azure Functions? Azure Functions is a serverless compute service in Microsoft Azure that allows developers to run small pieces of code without managing the underlying server infrastructure. Functions are commonly used for event-driven applications where code executes in response to events such as HTTP requests, timers, messages, and changes in Azure Storage. 2. Triggers A trigger defines the event that causes an Azure Function to execute. Common triggers include HTTP triggers, Timer triggers, Azure Blob Storage triggers, Azure Queue Storage triggers, and Azure Service Bus triggers. An HTTP trigger can expose a function through a REST endpoint. 3. Bindings Bindings provide a way for an Azure Function to connect to other Azure

Chunk ID : chunk_1
Source  : azure_function

In [15]:
# ============================================================
# CELL 14 - GENERATE GEMINI EMBEDDINGS FOR ALL CHUNKS
# ============================================================
# Each text chunk is converted into a numerical vector using:
#
#     gemini-embedding-001
#
# These vectors will be stored in ChromaDB.
#
# Later, when a user asks a question, we will also convert the
# question into an embedding and use ChromaDB to find the
# most semantically similar chunks.
# ============================================================


def create_embedding(text):
    """
    Generate a Gemini embedding for a piece of text.

    Parameters:
        text : Text to embed

    Returns:
        List of floating-point embedding values
    """

    response = gemini_client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )

    return response.embeddings[0].values


# ------------------------------------------------------------
# Generate embeddings for every chunk
# ------------------------------------------------------------

embeddings = []

print("Generating embeddings...")
print("-" * 60)

for index, chunk in enumerate(all_chunks):

    embedding = create_embedding(chunk["text"])

    embeddings.append(embedding)

    print(
        f"Chunk {index + 1}/{len(all_chunks)} "
        f"embedded successfully"
    )


# ------------------------------------------------------------
# Verify the embeddings
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EMBEDDING GENERATION COMPLETE")
print("=" * 60)

print("Total chunks :", len(all_chunks))
print("Total embeddings :", len(embeddings))

if embeddings:
    print("Embedding dimensions :", len(embeddings[0]))
    print("First 5 values :", embeddings[0][:5])

Generating embeddings...
------------------------------------------------------------
Chunk 1/7 embedded successfully
Chunk 2/7 embedded successfully
Chunk 3/7 embedded successfully
Chunk 4/7 embedded successfully
Chunk 5/7 embedded successfully
Chunk 6/7 embedded successfully
Chunk 7/7 embedded successfully

EMBEDDING GENERATION COMPLETE
Total chunks : 7
Total embeddings : 7
Embedding dimensions : 3072
First 5 values : [0.0056090886, -3.556526e-05, 0.025727497, -0.07020696, 0.0017520741]


In [16]:
# ============================================================
# CELL 15 - CREATE CHROMADB VECTOR DATABASE
# ============================================================
# ChromaDB will store:
#
#     Chunk text
#     Gemini embedding
#     Document metadata
#
# ChromaDB will persist the data inside our project's
# "chroma_db" folder.
# ============================================================


# ------------------------------------------------------------
# Create a persistent ChromaDB client
# ------------------------------------------------------------

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)


# ------------------------------------------------------------
# Create or load our collection
# ------------------------------------------------------------

collection = chroma_client.get_or_create_collection(
    name="azure_rag_collection"
)


# ------------------------------------------------------------
# Display collection information
# ------------------------------------------------------------

print("ChromaDB initialized successfully!")
print("Collection name:", collection.name)
print("Collection path:", CHROMA_DIR)
print("Existing documents:", collection.count())

ChromaDB initialized successfully!
Collection name: azure_rag_collection
Collection path: C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop\chroma_db
Existing documents: 0


In [17]:
# ============================================================
# CELL 16 - PREPARE CHUNKS AND METADATA FOR CHROMADB
# ============================================================
# ChromaDB requires:
#
#     IDs
#     Documents
#     Embeddings
#     Metadata
#
# We prepare those four collections here.
# ============================================================


# ------------------------------------------------------------
# Unique IDs
# ------------------------------------------------------------

ids = [
    chunk["chunk_id"]
    for chunk in all_chunks
]


# ------------------------------------------------------------
# Chunk text
# ------------------------------------------------------------

documents = [
    chunk["text"]
    for chunk in all_chunks
]


# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

metadatas = [
    {
        "filename": chunk["filename"],
        "page_number": chunk["page_number"]
    }
    for chunk in all_chunks
]


# ------------------------------------------------------------
# Verify the prepared data
# ------------------------------------------------------------

print("ChromaDB data prepared successfully!")

print("\nNumber of IDs       :", len(ids))
print("Number of documents :", len(documents))
print("Number of embeddings:", len(embeddings))
print("Number of metadata  :", len(metadatas))

print("\nExample metadata:")
print(metadatas[0])

ChromaDB data prepared successfully!

Number of IDs       : 7
Number of documents : 7
Number of embeddings: 7
Number of metadata  : 7

Example metadata:
{'filename': 'azure_functions.pdf', 'page_number': 1}


In [18]:
# ============================================================
# CELL 17 - INSERT DOCUMENTS INTO CHROMADB
# ============================================================
# This cell stores our RAG knowledge base in ChromaDB.
#
# For every chunk, ChromaDB stores:
#
#   ID          → Unique identifier
#   Document    → Actual text chunk
#   Embedding   → Gemini vector
#   Metadata    → PDF filename and page number
#
# ChromaDB persists this information inside:
#
#   C:\Users\SEJAL\RAG_WITH_GPT4All_Desktop\chroma_db
# ============================================================


# ------------------------------------------------------------
# Insert the chunks into ChromaDB
# ------------------------------------------------------------

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)


# ------------------------------------------------------------
# Verify the insertion
# ------------------------------------------------------------

print("=" * 60)
print("CHROMADB INSERTION COMPLETE")
print("=" * 60)

print("Documents stored in ChromaDB:", collection.count())

CHROMADB INSERTION COMPLETE
Documents stored in ChromaDB: 7


In [19]:
# ============================================================
# CELL 18 - VERIFY CHROMADB CONTENT
# ============================================================
# Retrieve a few records from ChromaDB to confirm that the
# documents, embeddings, and metadata were stored correctly.
# ============================================================


# Retrieve all stored records
stored_data = collection.get(
    include=["documents", "metadatas", "embeddings"]
)


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("=" * 60)
print("CHROMADB VERIFICATION")
print("=" * 60)

print("Number of stored IDs :", len(stored_data["ids"]))
print(
    "Number of documents  :",
    len(stored_data["documents"])
)
print(
    "Number of metadata   :",
    len(stored_data["metadatas"])
)
print(
    "Number of embeddings :",
    len(stored_data["embeddings"])
)


# ------------------------------------------------------------
# Display the first stored record
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FIRST STORED RECORD")
print("=" * 60)

print("ID:")
print(stored_data["ids"][0])

print("\nMetadata:")
print(stored_data["metadatas"][0])

print("\nDocument:")
print(stored_data["documents"][0][:500])

print("\nEmbedding dimensions:")
print(len(stored_data["embeddings"][0]))

CHROMADB VERIFICATION
Number of stored IDs : 7
Number of documents  : 7
Number of metadata   : 7
Number of embeddings : 7

FIRST STORED RECORD
ID:
chunk_0

Metadata:
{'filename': 'azure_functions.pdf', 'page_number': 1}

Document:
Azure Functions Serverless Compute on Microsoft Azure 1. What is Azure Functions? Azure Functions is a serverless compute service in Microsoft Azure that allows developers to run small pieces of code without managing the underlying server infrastructure. Functions are commonly used for event-driven applications where code executes in response to events such as HTTP requests, timers, messages, and changes in Azure Storage. 2. Triggers A trigger defines the event that causes an Azure Function to e

Embedding dimensions:
3072


In [20]:
# ============================================================
# CELL 19 - SEMANTIC SEARCH / DOCUMENT RETRIEVAL
# ============================================================
# This is the RETRIEVAL part of our RAG pipeline.
#
# Process:
#
# User Question
#       ↓
# Gemini Embedding
#       ↓
# Question Vector
#       ↓
# ChromaDB Similarity Search
#       ↓
# Most Relevant Document Chunks
#
# The retrieved chunks will later be passed to
# GPT4All Desktop as context.
# ============================================================


def retrieve_documents(
    question,
    top_k=3
):
    """
    Retrieve the most relevant document chunks from ChromaDB.

    Parameters:
        question : User's question
        top_k    : Number of relevant chunks to retrieve

    Returns:
        List of retrieved document records
    """

    # --------------------------------------------------------
    # Convert the question into a Gemini embedding
    # --------------------------------------------------------

    question_embedding = create_embedding(question)


    # --------------------------------------------------------
    # Search ChromaDB
    # --------------------------------------------------------

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )


    # --------------------------------------------------------
    # Convert ChromaDB response into a simple list
    # --------------------------------------------------------

    retrieved_documents = []

    for i in range(len(results["documents"][0])):

        retrieved_documents.append({
            "text": results["documents"][0][i],
            "filename": results["metadatas"][0][i]["filename"],
            "page_number": results["metadatas"][0][i]["page_number"],
            "distance": results["distances"][0][i]
        })


    return retrieved_documents

In [21]:
# ============================================================
# CELL 20 - TEST SEMANTIC RETRIEVAL
# ============================================================
# We will test whether ChromaDB can identify the relevant
# Azure Functions information for a user question.
# ============================================================


question = "What are the triggers supported by Azure Functions?"


# Retrieve the most relevant chunks
retrieved_documents = retrieve_documents(
    question,
    top_k=3
)


# ------------------------------------------------------------
# Display retrieval results
# ------------------------------------------------------------

print("=" * 80)
print("USER QUESTION")
print("=" * 80)

print(question)


print("\n" + "=" * 80)
print("RETRIEVED DOCUMENTS")
print("=" * 80)


for index, document in enumerate(
    retrieved_documents,
    start=1
):

    print(f"\n--- Result {index} ---")

    print("Source   :", document["filename"])
    print("Page     :", document["page_number"])
    print("Distance :", round(document["distance"], 4))

    print("\nText:")
    print(document["text"])

USER QUESTION
What are the triggers supported by Azure Functions?

RETRIEVED DOCUMENTS

--- Result 1 ---
Source   : azure_functions.pdf
Page     : 1
Distance : 0.444

Text:
Azure Functions Serverless Compute on Microsoft Azure 1. What is Azure Functions? Azure Functions is a serverless compute service in Microsoft Azure that allows developers to run small pieces of code without managing the underlying server infrastructure. Functions are commonly used for event-driven applications where code executes in response to events such as HTTP requests, timers, messages, and changes in Azure Storage. 2. Triggers A trigger defines the event that causes an Azure Function to execute. Common triggers include HTTP triggers, Timer triggers, Azure Blob Storage triggers, Azure Queue Storage triggers, and Azure Service Bus triggers. An HTTP trigger can expose a function through a REST endpoint. 3. Bindings Bindings provide a way for an Azure Function to connect to other Azure

--- Result 2 ---
Source   

In [22]:
# ============================================================
# CELL 21 - TEST RETRIEVAL FOR AZURE LOGIC APPS
# ============================================================

question_2 = "What are connectors in Azure Logic Apps?"


retrieved_documents_2 = retrieve_documents(
    question_2,
    top_k=3
)


print("=" * 80)
print("USER QUESTION")
print("=" * 80)

print(question_2)


print("\n" + "=" * 80)
print("RETRIEVED DOCUMENTS")
print("=" * 80)


for index, document in enumerate(
    retrieved_documents_2,
    start=1
):

    print(f"\n--- Result {index} ---")

    print("Source   :", document["filename"])
    print("Page     :", document["page_number"])
    print("Distance :", round(document["distance"], 4))

    print("\nText:")
    print(document["text"])

USER QUESTION
What are connectors in Azure Logic Apps?

RETRIEVED DOCUMENTS

--- Result 1 ---
Source   : azure_logic_apps.pdf
Page     : 1
Distance : 0.4572

Text:
queue. 3. Triggers Logic Apps support many trigger types. A workflow can start when an HTTP request is received, when a scheduled time is reached, when a message arrives, when a file is created, or when another connected service produces an event. 4. Connectors Connectors allow Logic Apps to communicate with other applications and services. Microsoft provides connectors for services such as Azure Service Bus, Blob Storage, SQL Server, Microsoft 365, SharePoint, Salesforce, and many other systems. Connectors simplify integration by providing predefined operations. 5. Azure Logic Apps Standard and Consumption Azure Logic Apps provides Standard and Consumption hosting options. Consumption workflows are commonly used for individual workflows that use a multitenant model. Standard supports work

--- Result 2 ---
Source   : azure_

In [23]:
# ============================================================
# CELL 22 - COMPLETE RAG PIPELINE
# ============================================================
# This function combines all the major components of our RAG
# application:
#
#     1. User Question
#             ↓
#     2. Gemini Question Embedding
#             ↓
#     3. ChromaDB Similarity Search
#             ↓
#     4. Retrieve Relevant PDF Chunks
#             ↓
#     5. Build Context
#             ↓
#     6. Send Context + Question to GPT4All Desktop
#             ↓
#     7. Phi-3 Mini Instruct generates the answer
#
# The model is instructed to answer ONLY from the retrieved
# context to reduce hallucinations.
# ============================================================


def ask_rag_question(
    question,
    top_k=3,
    max_tokens=400,
    temperature=0.2
):
    """
    Complete Retrieval-Augmented Generation pipeline.

    Parameters:
        question      : User's question
        top_k         : Number of ChromaDB chunks to retrieve
        max_tokens    : Maximum answer length
        temperature   : LLM generation temperature

    Returns:
        Dictionary containing:
            answer
            retrieved_documents
    """


    # --------------------------------------------------------
    # STEP 1 - Retrieve relevant documents
    # --------------------------------------------------------

    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )


    # --------------------------------------------------------
    # STEP 2 - Build the context
    # --------------------------------------------------------
    # Each retrieved chunk is included together with its
    # source filename and page number.
    # --------------------------------------------------------

    context_parts = []

    for document in retrieved_documents:

        context_parts.append(
            f"[Source: {document['filename']}, "
            f"Page: {document['page_number']}]\n"
            f"{document['text']}"
        )


    context = "\n\n".join(context_parts)


    # --------------------------------------------------------
    # STEP 3 - Create the RAG prompt
    # --------------------------------------------------------
    # The model is explicitly instructed not to use outside
    # knowledge.
    # --------------------------------------------------------

    prompt = f"""
You are a helpful technical assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT below.

Do not use outside knowledge.

If the answer cannot be determined from the context,
say:

"I could not find enough information in the provided
documents to answer this question."

Do not invent facts.

When appropriate, mention the source document and page
number.

-------------------- CONTEXT --------------------

{context}

------------------ END CONTEXT ------------------

USER QUESTION:
{question}

ANSWER:
"""


    # --------------------------------------------------------
    # STEP 4 - Prepare GPT4All Desktop API request
    # --------------------------------------------------------

    payload = {
        "model": GPT4ALL_MODEL_NAME,

        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],

        "max_tokens": max_tokens,
        "temperature": temperature
    }


    # --------------------------------------------------------
    # STEP 5 - Send request to GPT4All Desktop
    # --------------------------------------------------------

    try:

        response = requests.post(
            GPT4ALL_API_URL,
            json=payload,
            timeout=180
        )

        response.raise_for_status()

        result = response.json()


        # ----------------------------------------------------
        # STEP 6 - Extract generated answer
        # ----------------------------------------------------

        answer = result["choices"][0]["message"]["content"].strip()


    except requests.exceptions.RequestException as e:

        raise RuntimeError(
            f"GPT4All Desktop API request failed: {e}"
        )

    except (KeyError, IndexError, TypeError) as e:

        raise RuntimeError(
            f"Unexpected GPT4All Desktop response format: {e}"
        )


    # --------------------------------------------------------
    # STEP 7 - Return answer and retrieval information
    # --------------------------------------------------------

    return {
        "answer": answer,
        "retrieved_documents": retrieved_documents
    }


print("Complete RAG pipeline created successfully!")

Complete RAG pipeline created successfully!


In [24]:
# ============================================================
# CELL 23 - TEST COMPLETE RAG APPLICATION
# ============================================================

question = "What are the triggers supported by Azure Functions?"


# Run the complete RAG pipeline
rag_result = ask_rag_question(
    question,
    top_k=3
)


# ------------------------------------------------------------
# Display the answer
# ------------------------------------------------------------

print("=" * 80)
print("USER QUESTION")
print("=" * 80)

print(question)


print("\n" + "=" * 80)
print("GPT4ALL DESKTOP RAG ANSWER")
print("=" * 80)

print(rag_result["answer"])


# ------------------------------------------------------------
# Display retrieved sources
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)


for index, document in enumerate(
    rag_result["retrieved_documents"],
    start=1
):

    print(
        f"{index}. "
        f"{document['filename']} "
        f"(Page {document['page_number']})"
    )

USER QUESTION
What are the triggers supported by Azure Functions?

GPT4ALL DESKTOP RAG ANSWER
Azure Functions supports various triggers, including HTTP triggers (Page: 1), Timer triggers, Azure Blob Storage triggers, Azure Queue Storage triggers, and Azure Service Bus triggers.

Reference(s): [Source: azure_functions.pdf, Page: 2]

RETRIEVED SOURCES
1. azure_functions.pdf (Page 1)
2. azure_functions.pdf (Page 1)
3. azure_logic_apps.pdf (Page 1)


In [26]:
# ============================================================
# CELL 24 - CROSS-DOCUMENT RAG TEST
# ============================================================
# This test asks a question involving both Azure Functions
# and Azure Logic Apps.
#
# It demonstrates that the RAG system can retrieve relevant
# information from multiple documents.
# ============================================================


question = (
    "How can Azure Logic Apps and Azure Functions "
    "be used together?"
)


rag_result = ask_rag_question(
    question,
    top_k=4
)


# ------------------------------------------------------------
# Display the generated answer
# ------------------------------------------------------------

print("=" * 80)
print("USER QUESTION")
print("=" * 80)

print(question)


print("\n" + "=" * 80)
print("GPT4ALL DESKTOP RAG ANSWER")
print("=" * 80)

print(rag_result["answer"])


# ------------------------------------------------------------
# Display the retrieved sources
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)


for index, document in enumerate(
    rag_result["retrieved_documents"],
    start=1
):

    print(
        f"{index}. "
        f"{document['filename']} "
        f"(Page {document['page_number']})"
    )

USER QUESTION
How can Azure Logic Apps and Azure Functions be used together?

GPT4ALL DESKTOP RAG ANSWER
Azure Logic Apps and Azure Functions can be used together by utilizing a Logic App to call an Azure Function when custom processing or specialized business logic is required. This integration allows for workflow orchestration with service integrations using Logic Apps, while executing custom code in response to events through Azure Functions [Source: azure_logic_apps.pdf, Page: 1].

RETRIEVED SOURCES
1. azure_logic_apps.pdf (Page 1)
2. azure_logic_apps.pdf (Page 1)
3. azure_logic_apps.pdf (Page 1)
4. azure_functions.pdf (Page 1)


In [27]:
# ============================================================
# CELL 25 - INTERACTIVE RAG CHATBOT
# ============================================================
# This creates an interactive command-line style chatbot
# inside the Jupyter notebook.
#
# The user can continuously ask questions about the two
# PDF knowledge documents.
#
# Type "exit", "quit", or "q" to stop the chatbot.
# ============================================================


print("=" * 80)
print("RAG CHATBOT - GPT4ALL DESKTOP")
print("=" * 80)

print("Knowledge Base:")
print("  1. azure_functions.pdf")
print("  2. azure_logic_apps.pdf")

print("\nType your question below.")
print("Type 'exit' to stop the chatbot.")
print("=" * 80)


while True:

    # --------------------------------------------------------
    # Get question from the user
    # --------------------------------------------------------

    question = input("\nYou: ").strip()


    # --------------------------------------------------------
    # Exit condition
    # --------------------------------------------------------

    if question.lower() in ["exit", "quit", "q"]:
        print("\nChatbot stopped.")
        break


    # --------------------------------------------------------
    # Ignore empty questions
    # --------------------------------------------------------

    if not question:
        print("Please enter a question.")
        continue


    # --------------------------------------------------------
    # Run the complete RAG pipeline
    # --------------------------------------------------------

    try:

        result = ask_rag_question(
            question,
            top_k=3,
            max_tokens=400,
            temperature=0.2
        )


        # ----------------------------------------------------
        # Display generated answer
        # ----------------------------------------------------

        print("\nGPT4All: ")
        print(result["answer"])


        # ----------------------------------------------------
        # Display retrieved sources
        # ----------------------------------------------------

        print("\nSources:")

        # Avoid displaying the same source/page repeatedly
        displayed_sources = set()

        for document in result["retrieved_documents"]:

            source = (
                document["filename"],
                document["page_number"]
            )

            if source not in displayed_sources:

                print(
                    f"- {source[0]} "
                    f"(Page {source[1]})"
                )

                displayed_sources.add(source)


    except Exception as e:

        print("\nError while processing the question:")
        print(e)

RAG CHATBOT - GPT4ALL DESKTOP
Knowledge Base:
  1. azure_functions.pdf
  2. azure_logic_apps.pdf

Type your question below.
Type 'exit' to stop the chatbot.



You:  What is Azure Kubernetes Service?



GPT4All: 
I could not find enough information in the provided documents to answer this question.

Sources:
- azure_functions.pdf (Page 1)



You:  exit



Chatbot stopped.


In [ ]:
# ============================================================
# CELL 26 - RAG EVALUATION WITH MULTIPLE TEST QUESTIONS
# ============================================================
# This cell evaluates our RAG system using multiple questions.
#
# We test:
#
# 1. Azure Functions questions
# 2. Azure Logic Apps questions
# 3. Cross-document question
# 4. Out-of-scope question
#
# The results will later be used in the separate project
# documentation/report.
# ============================================================


evaluation_questions = [

    {
        "id": 1,
        "question": "What is Azure Functions?",
        "expected_source": "azure_functions.pdf"
    },

    {
        "id": 2,
        "question": "What are the triggers supported by Azure Functions?",
        "expected_source": "azure_functions.pdf"
    },

    {
        "id": 3,
        "question": "What are bindings in Azure Functions?",
        "expected_source": "azure_functions.pdf"
    },

    {
        "id": 4,
        "question": "What are Azure Logic Apps?",
        "expected_source": "azure_logic_apps.pdf"
    },

    {
        "id": 5,
        "question": "What are connectors in Azure Logic Apps?",
        "expected_source": "azure_logic_apps.pdf"
    },

    {
        "id": 6,
        "question": (
            "How can Azure Logic Apps and Azure Functions "
            "be used together?"
        ),
        "expected_source": "Both documents"
    },

    {
        "id": 7,
        "question": "What is Azure Kubernetes Service?",
        "expected_source": "Not available"
    }
]


# ------------------------------------------------------------
# Store evaluation results
# ------------------------------------------------------------

evaluation_results = []


# ------------------------------------------------------------
# Run every test question
# ------------------------------------------------------------

print("=" * 100)
print("RAG EVALUATION")
print("=" * 100)


for test in evaluation_questions:

    print("\n" + "=" * 100)

    print(
        f"TEST {test['id']}: "
        f"{test['question']}"
    )

    print("=" * 100)


    try:

        # ----------------------------------------------------
        # Run the complete RAG pipeline
        # ----------------------------------------------------

        result = ask_rag_question(
            test["question"],
            top_k=3,
            max_tokens=400,
            temperature=0.2
        )


        # ----------------------------------------------------
        # Extract retrieved sources
        # ----------------------------------------------------

        retrieved_sources = []

        for document in result["retrieved_documents"]:

            source = document["filename"]

            if source not in retrieved_sources:

                retrieved_sources.append(source)


        # ----------------------------------------------------
        # Determine whether expected source was retrieved
        # ----------------------------------------------------

        if test["expected_source"] == "Both documents":

            source_retrieved = (
                "azure_functions.pdf" in retrieved_sources
                and
                "azure_logic_apps.pdf" in retrieved_sources
            )

        elif test["expected_source"] == "Not available":

            # For an out-of-scope question we do not require
            # either document to be the expected source.
            source_retrieved = True

        else:

            source_retrieved = (
                test["expected_source"]
                in retrieved_sources
            )


        # ----------------------------------------------------
        # Store the result
        # ----------------------------------------------------

        evaluation_results.append({

            "test_id": test["id"],

            "question": test["question"],

            "expected_source": test["expected_source"],

            "retrieved_sources": ", ".join(
                retrieved_sources
            ),

            "source_retrieved": source_retrieved,

            "answer": result["answer"]

        })


        # ----------------------------------------------------
        # Display result
        # ----------------------------------------------------

        print("\nANSWER:")
        print(result["answer"])

        print("\nRETRIEVED SOURCES:")

        for source in retrieved_sources:
            print("-", source)

        print(
            "\nExpected Source:",
            test["expected_source"]
        )

        print(
            "Source Retrieval Match:",
            "PASS" if source_retrieved else "REVIEW"
        )


    except Exception as e:

        print("\nERROR:")
        print(e)


        evaluation_results.append({

            "test_id": test["id"],

            "question": test["question"],

            "expected_source": test["expected_source"],

            "retrieved_sources": "",

            "source_retrieved": False,

            "answer": f"ERROR: {e}"

        })


# ------------------------------------------------------------
# Evaluation summary
# ------------------------------------------------------------

print("\n\n" + "=" * 100)
print("EVALUATION COMPLETE")
print("=" * 100)

print(
    "Total test cases:",
    len(evaluation_results)
)

print(
    "Source retrieval matches:",
    sum(
        result["source_retrieved"]
        for result in evaluation_results
    )
)

print("\nEvaluation results are stored in:")
print("evaluation_results")

In [ ]:
# ============================================================
# CELL 27 - DISPLAY EVALUATION RESULTS AS A TABLE
# ============================================================

import pandas as pd


evaluation_df = pd.DataFrame(
    evaluation_results
)


# ------------------------------------------------------------
# Display selected evaluation information
# ------------------------------------------------------------

display(
    evaluation_df[
        [
            "test_id",
            "question",
            "expected_source",
            "retrieved_sources",
            "source_retrieved"
        ]
    ]
)

In [30]:
# ============================================================
# CELL 28 - CALCULATE BASIC RETRIEVAL METRICS
# ============================================================


total_tests = len(evaluation_results)


successful_retrievals = sum(
    result["source_retrieved"]
    for result in evaluation_results
)


retrieval_accuracy = (
    successful_retrievals / total_tests * 100
    if total_tests > 0
    else 0
)


print("=" * 60)
print("RAG EVALUATION METRICS")
print("=" * 60)

print("Total test cases       :", total_tests)

print(
    "Successful retrievals  :",
    successful_retrievals
)

print(
    "Retrieval match rate   :",
    f"{retrieval_accuracy:.2f}%"
)

RAG EVALUATION METRICS
Total test cases       : 7
Successful retrievals  : 4
Retrieval match rate   : 57.14%


In [31]:
# ============================================================
# CELL 29 - DETAILED RAG EVALUATION RESULTS
# ============================================================

print("=" * 100)
print("DETAILED RAG EVALUATION RESULTS")
print("=" * 100)

for result in evaluation_results:

    status = (
        "PASS"
        if result["source_retrieved"]
        else "REVIEW"
    )

    print("\n" + "-" * 100)

    print(f"Test ID          : {result['test_id']}")
    print(f"Question         : {result['question']}")
    print(f"Expected Source  : {result['expected_source']}")
    print(f"Retrieved Source : {result['retrieved_sources']}")
    print(f"Status           : {status}")

    print("\nAnswer:")
    print(result["answer"])

DETAILED RAG EVALUATION RESULTS

----------------------------------------------------------------------------------------------------
Test ID          : 1
Question         : What is Azure Functions?
Expected Source  : azure_functions.pdf
Retrieved Source : azure_functions.pdf
Status           : PASS

Answer:
Azure Functions is a serverless compute service in Microsoft Azure that allows developers to run small pieces of code without managing the underlying server infrastructure. It's commonly used for event-driven applications where code executes in response to events such as HTTP requests, timers, messages, and changes in Azure Storage [Source: azure_functions.pdf, Page: 1].

----------------------------------------------------------------------------------------------------
Test ID          : 2
Question         : What are the triggers supported by Azure Functions?
Expected Source  : azure_functions.pdf
Retrieved Source : azure_functions.pdf, azure_logic_apps.pdf
Status           : PAS